# 생각중인 요소
1. Tableau용 파생 컬럼 추가
    - member_year
    - member_month
    - member_quarter
    - age_group
    - income_group

# 데이터 전처리

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 사용 함수

In [2]:
# 컬럼 정보 간단 표현
def check_basic_info(df, df_name, exclude_cols=None):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 컬럼 정보 / 결측치 확인 정보 요약")
    print(f"{'='*80}\n")


    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)
    
    # 1. 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })
    
    # 2. 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    })
    
    # 3. 보기 좋게 정렬
    summary_df = summary_df.sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )
    
    print("[전체 요약]")
    display(overview_df)
    
    print("[컬럼별 요약]")
    display(summary_df)

    print("[테이블 요약]")
    display(df.head())

In [3]:
# 중복값 확인
def check_id_duplicates(df, col_name, df_name):
    df_copied = df.copy()
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 값 중복 확인")
    print(f"{'='*80}")
    
    print("전체 행 수:", len(df_copied))
    print(f"{col_name} 고유 개수:", df_copied[col_name].nunique())
    print(f"중복 {col_name} 개수:", df_copied[col_name].duplicated().sum())
    
    display(df_copied.head())

In [4]:
# 컬럼 분포 확인
def check_category_summary(df, df_name, col_name):
    df_copied = df.copy()
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")
    
    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return
    
    summary_df = df_copied[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, '개수']
    summary_df['비율(%)'] = (summary_df['개수'] / len(df_copied) * 100).round(2)
    
    print("전체 행 수:", len(df_copied))
    print(f"{col_name} 고유값 개수(결측 포함):", df_copied[col_name].nunique(dropna=False))
    print()
    
    display(summary_df.head(10))

# 테이블 전처리

In [5]:
# 1. 원본 데이터 로드
df_portfolio = pd.read_csv("data/portfolio.csv", index_col=0)
df_profile = pd.read_csv("data/profile.csv", index_col=0)
df_transcript = pd.read_csv("data/transcript.csv", index_col=0)

## 1. df_portfolio 전처리

In [6]:
# 확인 및 복제
print("[portfolio]")
df_portfolio_copy = df_portfolio.copy()
check_basic_info(df_portfolio_copy, "portfolio")

[portfolio]

portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
id,str,10,100.0,0,0.0,10
reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3


[테이블 요약]


,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [7]:
# 컬럼명 정리
# portfolio의 id는 고객 id가 아니라 프로모션 식별자이므로 offer_id로 변경
# reward는 오퍼의 보상값 의미를 명확히 하기 위해 offer_reward로 변경
df_portfolio_clean = (
    df_portfolio_copy
    .rename(columns={
        'id': 'offer_id',           # 프로모션(offer)을 구분하는 식별자
        'reward': 'offer_reward'    # 오퍼 보상값
    })
)

display(df_portfolio_clean.head())

,offer_reward,channels,difficulty,duration,offer_type,offer_id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [8]:
# channels 문자열에서 채널 포함 여부를 더미 컬럼으로 생성
channel_list = ['web', 'email', 'mobile', 'social']

for ch in channel_list:
    df_portfolio_clean[ch] = (
        df_portfolio_clean['channels']
        .str.contains(ch, na=False) # 결측치가 있을경우 False로 처리해 오류 방지
        .astype(int)
    )

display(df_portfolio_clean.head())

,offer_reward,channels,difficulty,duration,offer_type,offer_id,web,email,mobile,social
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0


In [9]:
# 컬럼 순서 정리
# 식별자(offer_id) → 오퍼 속성(offer_type, offer_reward, difficulty, duration) 
# → 채널 정보(channels, web, email, mobile, social) 순으로 배치
df_portfolio_clean = df_portfolio_clean[
    [
        'offer_id',
        'offer_type', 'offer_reward', 'difficulty', 'duration',
        'channels',
        'web', 'email', 'mobile', 'social'
    ]
]

## df_portfolio 전처리 최종 확인

In [10]:
check_basic_info(df_portfolio_clean, "portfolio_clean")
check_id_duplicates(df_portfolio_clean, 'offer_id', 'portfolio_clean')


portfolio_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,10
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
offer_id,str,10,100.0,0,0.0,10
offer_reward,int64,10,100.0,0,0.0,5
difficulty,int64,10,100.0,0,0.0,5
duration,int64,10,100.0,0,0.0,5
channels,str,10,100.0,0,0.0,4
offer_type,str,10,100.0,0,0.0,3
web,int64,10,100.0,0,0.0,2
mobile,int64,10,100.0,0,0.0,2
social,int64,10,100.0,0,0.0,2
email,int64,10,100.0,0,0.0,1


[테이블 요약]


,offer_id,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social
0,ae264e3637204a6fb9bb56bc8210ddfd,bogo,10,10,7,"['email', 'mobile', 'social']",0,1,1,1
1,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10,10,5,"['web', 'email', 'mobile', 'social']",1,1,1,1
2,3f207df678b143eea3cee63160fa8bed,informational,0,0,4,"['web', 'email', 'mobile']",1,1,1,0
3,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5,5,7,"['web', 'email', 'mobile']",1,1,1,0
4,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,5,20,10,"['web', 'email']",1,1,0,0



portfolio_clean의 값 중복 확인
전체 행 수: 10
offer_id 고유 개수: 10
중복 offer_id 개수: 0


,offer_id,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social
0,ae264e3637204a6fb9bb56bc8210ddfd,bogo,10,10,7,"['email', 'mobile', 'social']",0,1,1,1
1,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10,10,5,"['web', 'email', 'mobile', 'social']",1,1,1,1
2,3f207df678b143eea3cee63160fa8bed,informational,0,0,4,"['web', 'email', 'mobile']",1,1,1,0
3,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5,5,7,"['web', 'email', 'mobile']",1,1,1,0
4,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,5,20,10,"['web', 'email']",1,1,0,0


## 2. df_profile 전처리

In [11]:
# 확인 및 복제
print("\n[profile]")
df_profile_copy = df_profile.copy()
check_basic_info(df_profile_copy, "profile")


[profile]

profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,17000
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
income,float64,14825,87.21,2175,12.79,91
gender,str,14825,87.21,2175,12.79,3
id,str,17000,100.00,0,0.00,17000
became_member_on,int64,17000,100.00,0,0.00,1716
age,int64,17000,100.00,0,0.00,85


[테이블 요약]


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


In [12]:
# 컬럼명 정리
df_profile_copy = (
    df_profile_copy
    .rename(columns={
    'id': 'customer_id'
}))

In [13]:
# 가입일 날짜형 변환
became_member_str = df_profile_copy['became_member_on'].astype(str)

df_profile_copy['became_member_on'] = pd.to_datetime(
    became_member_str,
    format='%Y%m%d'
)

In [14]:
# 가입일 파생 컬럼

# 파생컬럼 필요시 사용
# # 가입 년도
# df_profile_copy['member_year'] = (
#     df_profile_copy['became_member_on']
#     .dt
#     .year
# )

# # 가입 월
# df_profile_copy['member_month'] = (
#     df_profile_copy['became_member_on'].
#     dt
#     .month
# )

# # 가입 분기
# df_profile_clean['member_quarter'] = (
#     df_profile_clean['became_member_on']
#     .dt
#     .quarter
# )

In [15]:
# age 이상치 처리
# age=118은 비정상값으로 보고 NaN 처리
before_len = len(df_profile_copy)

age_118_count = (df_profile_copy['age'] == 118).sum()
df_profile_copy['age'] = df_profile_copy['age'].replace(118, np.nan)

print("age=118 처리 건수:", age_118_count)
print()

# 주요 컬럼 결측치 현황 확인
age_missing = df_profile_copy['age'].isna().sum()
gender_missing = df_profile_copy['gender'].isna().sum()
income_missing = df_profile_copy['income'].isna().sum()

print("age 결측치 수:", age_missing)
print("gender 결측치 수:", gender_missing)
print("income 결측치 수:", income_missing)
print()

# 결측치 중복 여부 확인
age_gender_missing = ((                     # - age와 gender가 모두 결측인 행
    df_profile_copy["gender"].isnull() & 
    df_profile_copy["age"].isnull()).sum()
)     

age_income_missing = ((                     # - age와 income이 모두 결측인 행
    df_profile_copy["income"].isnull() & 
    df_profile_copy["age"].isnull()).sum()
)     

age_gender_income_missing = ((              # - age, gender, income이 모두 결측인 행
    df_profile_copy["gender"].isnull() & 
    df_profile_copy["income"].isnull() & 
    df_profile_copy["age"].isnull())
    .sum()
)

print("gender 결측이면서 age도 결측인 행 수:", age_gender_missing)
print("income 결측이면서 age도 결측인 행 수:", age_income_missing)
print("gender, income, age가 모두 결측인 행 수:", age_gender_income_missing)

age=118 처리 건수: 2175

age 결측치 수: 2175
gender 결측치 수: 2175
income 결측치 수: 2175

gender 결측이면서 age도 결측인 행 수: 2175
income 결측이면서 age도 결측인 행 수: 2175
gender, income, age가 모두 결측인 행 수: 2175


In [16]:
# 주요 컬럼 결측치 제거
df_profile_copy = df_profile_copy.dropna(subset=['age', 'gender', 'income']).copy()

after_len = len(df_profile_copy)

print("제거 전 행 수:", before_len)
print("제거 후 행 수:", after_len)
print("제거된 행 수:", before_len - after_len)

제거 전 행 수: 17000
제거 후 행 수: 14825
제거된 행 수: 2175


In [17]:
# 컬럼 순서 정리
# 식별자(customer_id) → 고객 특성(gender, age, income) → 가입일(became_member_on) 순으로 배치
df_profile_clean = df_profile_copy[
    [
        'customer_id',
        'gender', 'age', 'income', 
        'became_member_on'
    ]
]

## df_profile 전처리 최종확인

In [18]:
check_basic_info(df_profile_clean, "profile_clean")
check_id_duplicates(df_profile_clean, 'customer_id', 'profile_clean')


profile_clean의 컬럼 정보 / 결측치 확인 정보 요약



[전체 요약]


,항목,값
0,행 개수,14825
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
customer_id,str,14825,100.0,0,0.0,14825
became_member_on,datetime64[us],14825,100.0,0,0.0,1707
income,float64,14825,100.0,0,0.0,91
age,float64,14825,100.0,0,0.0,84
gender,str,14825,100.0,0,0.0,3


[테이블 요약]


,customer_id,gender,age,income,became_member_on
1,0610b486422d4921ae7d2bf64640c50b,F,55.0,112000.0,2017-07-15
3,78afa995795e4d85b5d9ceeca43f5fef,F,75.0,100000.0,2017-05-09
5,e2127556f4f64592b11af22de27a7932,M,68.0,70000.0,2018-04-26
8,389bc3fa690240e798340f5a15918d5c,M,65.0,53000.0,2018-02-09
12,2eeac8d8feae4a8cad5a6af0499a211d,M,58.0,51000.0,2017-11-11



profile_clean의 값 중복 확인
전체 행 수: 14825
customer_id 고유 개수: 14825
중복 customer_id 개수: 0


,customer_id,gender,age,income,became_member_on
1,0610b486422d4921ae7d2bf64640c50b,F,55.0,112000.0,2017-07-15
3,78afa995795e4d85b5d9ceeca43f5fef,F,75.0,100000.0,2017-05-09
5,e2127556f4f64592b11af22de27a7932,M,68.0,70000.0,2018-04-26
8,389bc3fa690240e798340f5a15918d5c,M,65.0,53000.0,2018-02-09
12,2eeac8d8feae4a8cad5a6af0499a211d,M,58.0,51000.0,2017-11-11


## 3. df_transcript 전처리

In [19]:
# 확인 및 복제
print("[df_transcript]")
df_transcript_copy = df_transcript.copy()
check_basic_info(df_transcript_copy, "transcript")
check_category_summary(df_transcript_copy, "transcript", "event")

[df_transcript]

transcript의 컬럼 정보 / 결측치 확인 정보 요약



[전체 요약]


,항목,값
0,행 개수,306534
1,열 개수,4
2,중복 행 개수,397


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
person,str,306534,100.0,0,0.0,17000
value,str,306534,100.0,0,0.0,5121
time,int64,306534,100.0,0,0.0,120
event,str,306534,100.0,0,0.0,4


[테이블 요약]


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0



transcript의 event 범주 확인
전체 행 수: 306534
event 고유값 개수(결측 포함): 4



,event,개수,비율(%)
0,transaction,138953,45.33
1,offer received,76277,24.88
2,offer viewed,57725,18.83
3,offer completed,33579,10.95


In [20]:
# 컬럼명 정리
df_transcript_copy = (
    df_transcript_copy
    .rename(columns={
    'person': 'customer_id'
}))

In [21]:
# value 컬럼 파싱

# 문자열로 저장된 딕셔너리/리스트 등의 값을 실제 파이썬 객체로 변환하기 위해 사용
import ast

def safe_literal_eval(x):
    '''문자열 형태의 value 값을 실제 파이썬 객체로 안전하게 변환'''
    try:                                # 문자열이 아니면 원래 값을 그대로 반환
        return ast.literal_eval(x) if isinstance(x, str) else x
    except (ValueError, SyntaxError):   # 값 형식이 잘못되어 literal_eval이 실패하면 에러 대신 원래 값을 유지
        return x

# value 컬럼 파싱
df_transcript_copy['value'] = df_transcript_copy['value'].apply(safe_literal_eval)

# 변환 후 자료형 확인
print(df_transcript_copy['value'].apply(type).value_counts())

value
<class 'dict'>    306534
Name: count, dtype: int64


In [22]:
# value 컬럼 key 구조 및 빈도 확인
all_keys = [
    key
    for d in df_transcript_copy['value']
    for key in d.keys()
]

key_freq_df = (
    pd.Series(all_keys)
    .value_counts()
    .reset_index()
)

key_freq_df.columns = ['key', '빈도']

print("value 컬럼 전체 key 종류:")
print(sorted(key_freq_df['key'].tolist()))
print()

display(key_freq_df)

value 컬럼 전체 key 종류:
['amount', 'offer id', 'offer_id', 'reward']



,key,빈도
0,amount,138953
1,offer id,134002
2,offer_id,33579
3,reward,33579


In [23]:
# ============================================================
# value 컬럼(dict) 분리 및 주요 파생 컬럼 생성
# event별로 value 안에 들어 있는 값을 별도 컬럼으로 분리
# 'offer id'와 'offer_id'는 하나의 offer_id 컬럼으로 통합
# ============================================================
value_df = pd.DataFrame(df_transcript_copy['value'].tolist())
df_transcript_copy = pd.concat([df_transcript_copy, value_df], axis=1)

# offer id / offer_id 컬럼명을 하나로 통합
df_transcript_copy['offer_id'] = df_transcript_copy['offer id'].fillna(df_transcript_copy['offer_id'])

df_transcript_copy = (
    df_transcript_copy
    .rename(columns={
    'reward': 'event_reward'
    })
)

# 주요 컬럼 확인
display(
    df_transcript_copy[
        [
            'customer_id', 
            'event', 
            'time', 
            'offer_id', 
            'amount', 
            'event_reward']]
        .head()
)

,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


In [24]:
# event별 컬럼 결측 개수 확인
print("="*80)
print("이벤트별 결측 확인")
print("="*80)

print(df_transcript_copy[['event', 'offer_id', 'amount', 'event_reward']].isna().groupby(df_transcript_copy['event']).sum())

이벤트별 결측 확인
                 event  offer_id  amount  event_reward
event                                                 
offer completed      0         0   33579             0
offer received       0         0   76277         76277
offer viewed         0         0   57725         57725
transaction          0    138953       0        138953


In [25]:
# 중복 확인용 value 비교 컬럼 생성
df_transcript_copy['value_str'] = df_transcript_copy['value'].astype(str)

# 중복 기준
dup_cols = ['customer_id', 'event', 'time', 'offer_id', 'event_reward', 'value_str']

# 중복 행 추출
dup_df = df_transcript_copy[
    df_transcript_copy.duplicated(subset=dup_cols, keep=False)
].sort_values(['customer_id', 'time', 'offer_id'])

print("중복에 포함된 전체 행 수:", len(dup_df))
print()

print("[중복 event 분포]")
print(dup_df['event'].value_counts())
print()

print("[중복 횟수 분포]")
dup_count_summary = (
    df_transcript_copy
    .groupby(dup_cols)
    .size()
    .reset_index(name='dup_count')
    .query("dup_count > 1")
)

print(dup_count_summary['dup_count'].value_counts().sort_index())
print()

print("완전 동일 중복 그룹 수:", len(dup_count_summary))


중복에 포함된 전체 행 수: 793

[중복 event 분포]
event
offer completed    793
Name: count, dtype: int64

[중복 횟수 분포]
dup_count
2    395
3      1
Name: count, dtype: int64

완전 동일 중복 그룹 수: 396


중복은 `offer completed` 이벤트에서만 발생했다.

중복 기준 컬럼(customer_id, event, time, offer_id, event_reward, value_str)으로 확인한 결과,\
총 396개의 완전 동일 중복 그룹이 존재했다.\
이 중 395개 그룹은 2건 중복, 1개 그룹은 3건 중복으로, 중복에 포함된 전체 행 수는 793행이다.

따라서 해당 중복은 새로운 정보를 추가하지 않는 완전 동일 반복 행으로 판단하고\
각 그룹에서 1개만 남기고 나머지를 제거하였다.

In [26]:
# ============================================================
# 중복 제거
# 중복 기준 컬럼이 모두 동일한 완전 동일 중복 행이므로 제거
# 각 중복 그룹에서 1개는 유지하고 나머지만 삭제
# ============================================================
df_transcript_drop = (
    df_transcript_copy
    .drop_duplicates(subset=dup_cols)
    .drop(columns=['offer id', 'value_str'], errors='ignore')
    .copy()
)

print("중복 제거 전 행 수:", len(df_transcript_copy))
print("중복 제거 후 행 수:", len(df_transcript_drop))
print("제거된 행 수:", len(df_transcript_copy) - len(df_transcript_drop))

중복 제거 전 행 수: 306534
중복 제거 후 행 수: 306137
제거된 행 수: 397


실제 제거되는 행 수는 397행이다.\
이는 drop_duplicates()가 각 중복 그룹에서 1개 행은 유지하고, 나머지 중복 행만 제거하기 때문이다.

In [27]:
# ============================================================
# value 컬럼 드랍
# value 컬럼을 분리 및 주요 파생 컬럼 생성했기 때문에 제거
# ============================================================
df_transcript_drop = (
    df_transcript_drop
    .drop(columns=['value'], errors='ignore')
    .copy()
)

df_transcript_drop.columns

Index(['customer_id', 'event', 'time', 'amount', 'offer_id', 'event_reward'], dtype='str')

In [28]:
# 컬럼 순서 정리
# 식별자(customer_id) → 이벤트 정보(event, time) 
# → 오퍼/거래 관련 정보(offer_id, amount, event_reward) 순으로 배치
df_transcript_clean = df_transcript_drop[
    [
        'customer_id',
        'event','time',
        'offer_id','amount','event_reward'
    ]
]

## df_transcript 전처리 최종확인

In [29]:
check_basic_info(df_transcript_clean, "transcript_clean")
check_id_duplicates(df_transcript_clean, 'customer_id', 'transcript_clean(customer_id)')
check_id_duplicates(df_transcript_clean, 'offer_id', 'transcript_clean(offer_id)')


transcript_clean의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
customer_id,str,306137,100.00,0,0.00,17000
time,int64,306137,100.00,0,0.00,120
event,str,306137,100.00,0,0.00,4


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN



transcript_clean(customer_id)의 값 중복 확인
전체 행 수: 306137
customer_id 고유 개수: 17000
중복 customer_id 개수: 289137


,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN



transcript_clean(offer_id)의 값 중복 확인
전체 행 수: 306137
offer_id 고유 개수: 10
중복 offer_id 개수: 306126


,customer_id,event,time,offer_id,amount,event_reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


# 코드 통합

## transcript과 profile 매칭 확인

In [30]:
# 조인키: customer_id
transcript_customer_ids = set(df_transcript_clean['customer_id'].dropna())
profile_customer_ids = set(df_profile_clean['customer_id'].dropna())

print("transcript customer_id 고유 개수:", len(transcript_customer_ids))
print("profile customer_id 고유 개수:", len(profile_customer_ids))

unmatched_customers = transcript_customer_ids - profile_customer_ids
print("profile에 없는 transcript 고객 수:", len(unmatched_customers))

transcript customer_id 고유 개수: 17000
profile customer_id 고유 개수: 14825
profile에 없는 transcript 고객 수: 2175


## transcript과 portfolio 매칭 확인

In [31]:
# 조인키: offer_id
transcript_offer_ids = set(df_transcript_clean['offer_id'].dropna())
portfolio_offer_ids = set(df_portfolio_clean['offer_id'].dropna())

print("transcript 내 offer_id 개수:", len(transcript_offer_ids))
print("portfolio 내 offer_id 개수:", len(portfolio_offer_ids))
print("portfolio에 없는 transcript offer_id 수:", len(transcript_offer_ids - portfolio_offer_ids))
print("transcript에 없는 portfolio offer_id 수:", len(portfolio_offer_ids - transcript_offer_ids))

transcript 내 offer_id 개수: 10
portfolio 내 offer_id 개수: 10
portfolio에 없는 transcript offer_id 수: 0
transcript에 없는 portfolio offer_id 수: 0


## transcript + profile

In [32]:
# ============================================================
# transcript + profile 컬럼 조인
# transcript를 기준 테이블로 두고, 이벤트 로그 손실 없이 유지하기 위해 left join 사용
# profile에 존재하는 고객 정보만 customer_id를 조인키로 매칭해 추가
# ============================================================
df_transcript_profile = df_transcript_clean.merge(
    df_profile_clean,
    on='customer_id',
    how='left'
)

df_transcript_profile = df_transcript_profile[
    [
        # 로그 정보
        'customer_id', 'event', 'time', 'offer_id',

        # 이벤트/거래 정보
        'amount', 'event_reward',

        # 고객 정보
        'gender', 'age', 'income', 'became_member_on',
    ]
]


In [33]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("transcript + profile 행 수:", len(df_transcript_profile))
print("행 수 차이:", len(df_transcript_profile) - len(df_transcript_clean))

check_basic_info(df_transcript_profile, "transcript + profile", exclude_cols = 'value')
check_id_duplicates(df_transcript_profile, 'customer_id', 'transcript + profile')

transcript 원본 행 수: 306137
transcript + profile 행 수: 306137
행 수 차이: 0

transcript + profile의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
became_member_on,datetime64[us],272388,88.98,33749,11.02,1707
income,float64,272388,88.98,33749,11.02,91
age,float64,272388,88.98,33749,11.02,84
gender,str,272388,88.98,33749,11.02,3
customer_id,str,306137,100.00,0,0.00,17000
time,int64,306137,100.00,0,0.00,120
event,str,306137,100.00,0,0.00,4


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,gender,age,income,became_member_on
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,F,75.0,100000.0,2017-05-09
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,M,68.0,70000.0,2018-04-26
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN,NaN,NaN,NaT



transcript + profile의 값 중복 확인
전체 행 수: 306137
customer_id 고유 개수: 17000
중복 customer_id 개수: 289137


,customer_id,event,time,offer_id,amount,event_reward,gender,age,income,became_member_on
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,F,75.0,100000.0,2017-05-09
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,M,68.0,70000.0,2018-04-26
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN,NaN,NaN,NaT


offer_id, amount, event_reward의 결측은 이벤트 유형 차이에 따라 구조적으로 발생한 결측값이다.\

반면 gender, age, income, became_member_on의 결측은 profile 전처리 과정에서 일부 고객이 제거된 뒤 transcript와 left join 하면서 발생한 결측이다.\
따라서 고객 세그먼트 기반 분석에서는 해당 결측 행의 처리 기준을 별도로 명확히 할 필요가 있다.

## transcript + portfolio

In [34]:
# ============================================================
# transcript + portfolio 컬럼 조인
# transcript를 기준 테이블로 두고, 이벤트 로그 손실 없이 유지하기 위해 left join 사용
# portfolio에 존재하는 오퍼 정보만 offer_id를 조인키로 매칭해 추가
# ============================================================
df_transcript_portfolio = df_transcript_clean.merge(
    df_portfolio_clean,
    on='offer_id',
    how='left'
)

df_transcript_portfolio = df_transcript_portfolio[
    [
        # 로그 정보
        'customer_id', 'event', 'time', 'offer_id',

        # 이벤트/거래 정보
        'amount', 'event_reward',

        # 오퍼 정보
        'offer_reward', 'offer_type', 'difficulty', 'duration', 'channels',

        # 채널 정보
        'web', 'email', 'mobile', 'social',
    ]
]


In [35]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("transcript + portfolio 행 수:", len(df_transcript_portfolio))
print("행 수 차이:", len(df_transcript_portfolio) - len(df_transcript_clean))

check_basic_info(df_transcript_portfolio, "transcript + portfolio")
check_id_duplicates(df_transcript_portfolio, 'offer_id', 'transcript + portfolio')

transcript 원본 행 수: 306137
transcript + portfolio 행 수: 306137
행 수 차이: 0

transcript + portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,15
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
offer_reward,float64,167184,54.61,138953,45.39,5
difficulty,float64,167184,54.61,138953,45.39,5
duration,float64,167184,54.61,138953,45.39,5
channels,str,167184,54.61,138953,45.39,4
offer_type,str,167184,54.61,138953,45.39,3
web,float64,167184,54.61,138953,45.39,2
mobile,float64,167184,54.61,138953,45.39,2


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,offer_reward,offer_type,difficulty,duration,channels,web,email,mobile,social
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,5.0,bogo,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,5.0,discount,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,2.0,discount,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,2.0,discount,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,10.0,bogo,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0



transcript + portfolio의 값 중복 확인
전체 행 수: 306137
offer_id 고유 개수: 10
중복 offer_id 개수: 306126


,customer_id,event,time,offer_id,amount,event_reward,offer_reward,offer_type,difficulty,duration,channels,web,email,mobile,social
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,5.0,bogo,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,5.0,discount,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,2.0,discount,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,2.0,discount,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,10.0,bogo,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0


offer_id, offer_reward, offer_type, difficulty, duration, channels, web, email, mobile, social의 결측은\
offer_id가 없는 이벤트 유형에서 구조적으로 발생한 결측값이다.

또한 amount의 결측은 거래(transaction) 이벤트가 아닌 오퍼 이벤트에서 자연스럽게 발생한 결측이며,\
event_reward의 결측은 offer completed가 아닌 이벤트에서 구조적으로 발생한 결측이다.

따라서 조인 이후 나타난 결측은 조인 오류라기보다,\
이벤트 유형 차이에 따라 발생한 구조적 결측으로 해석할 수 있다.

## transcript + portfolio + profile

In [36]:
# ============================================================
# transcript + portfolio + profile 컬럼 조인
# transcript를 기준 테이블로 두고, 이벤트 로그 손실 없이 유지하기 위해 left join 사용
# offer_id를 조인키로 portfolio의 오퍼 정보를 추가,
# customer_id를 조인키로 profile의 고객 정보를 추가
# ============================================================
df_transcript_portfolio_profile = (
    df_transcript_clean
    .merge(df_portfolio_clean, on='offer_id', how='left')
    .merge(df_profile_clean, on='customer_id', how='left')
)

df_transcript_portfolio_profile = df_transcript_portfolio_profile[
    [
        # 로그 정보
        'customer_id', 'event', 'time', 'offer_id',

        # 이벤트/거래 정보
        'amount', 'event_reward',

        # 오퍼 정보
        'offer_type', 'offer_reward', 'difficulty', 'duration', 'channels',

        # 채널 정보
        'web', 'email', 'mobile', 'social',

        # 고객 정보
        'gender', 'age', 'income', 'became_member_on',
    ]
]

In [37]:
print("transcript 원본 행 수:", len(df_transcript_clean))
print("전체 통합 행 수:", len(df_transcript_portfolio_profile))
print("행 수 차이:", len(df_transcript_portfolio_profile) - len(df_transcript_clean))

check_basic_info(df_transcript_portfolio_profile, "transcript + portfolio + profile")
check_id_duplicates(df_transcript_portfolio_profile, 'customer_id', 'transcript + portfolio + profile (customer_id)')
check_id_duplicates(df_transcript_portfolio_profile, 'offer_id', 'transcript + portfolio + profile (offer_id)')

transcript 원본 행 수: 306137
전체 통합 행 수: 306137
행 수 차이: 0

transcript + portfolio + profile의 컬럼 정보 / 결측치 확인 정보 요약



[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,19
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
offer_reward,float64,167184,54.61,138953,45.39,5
difficulty,float64,167184,54.61,138953,45.39,5
duration,float64,167184,54.61,138953,45.39,5
channels,str,167184,54.61,138953,45.39,4
offer_type,str,167184,54.61,138953,45.39,3
web,float64,167184,54.61,138953,45.39,2
mobile,float64,167184,54.61,138953,45.39,2


...

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaT
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT



transcript + portfolio + profile (customer_id)의 값 중복 확인
전체 행 수: 306137
customer_id 고유 개수: 17000
중복 customer_id 개수: 289137


,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaT
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT



transcript + portfolio + profile (offer_id)의 값 중복 확인
전체 행 수: 306137
offer_id 고유 개수: 10
중복 offer_id 개수: 306126


,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaT
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaT


offer_id 관련 컬럼의 결측은 offer_id가 없는 이벤트에서 구조적으로 발생한 결측값이다.

또한 amount는 transaction 이벤트에서만,\
event_reward는 offer completed 이벤트에서만 값이 존재하므로 \
이 역시 이벤트 유형 차이에 따른 자연스러운 결측이다.

반면 gender, age, income, became_member_on의 결측은\
profile 전처리 과정에서 일부 고객이 제거된 뒤 left join 하면서 발생한 결측이다.

따라서 통합 테이블의 결측은\
이벤트 유형 차이에 따른 구조적 결측과\
profile 미매칭으로 인한 결측으로 구분해 해석할 필요가 있다.

## profile 정보가 없는 고객 이벤트

In [38]:
# ============================================================
# profile 전처리에서 제외된 고객은 left join 후
# 주요 고객 정보 컬럼이 모두 결측으로 나타나므로,
# 4개 컬럼이 모두 NaN인 경우를 profile 미매칭 고객으로 판단
# ============================================================
profile_cols = [col for col in ['gender', 'age', 'income', 'became_member_on']
                if col in df_transcript_profile.columns]

missing_profile_mask = df_transcript_profile[profile_cols].isna().all(axis=1)
df_profile_missing_events = df_transcript_profile.loc[missing_profile_mask].copy()

missing_event_count = len(df_profile_missing_events)
total_event_count = len(df_transcript_profile)
missing_event_ratio = (missing_event_count / total_event_count) * 100

print("=" * 60)
print("profile 정보가 없는 고객 이벤트 요약")
print("=" * 60)
print(f"사용한 profile 컬럼: {profile_cols}")
print(f"이벤트 행 수: {missing_event_count}")
print(f"고객 수: {df_profile_missing_events['customer_id'].nunique()}")
print()

print("[event별 개수]")
print(df_profile_missing_events['event'].value_counts(dropna=False))
print()

print("=" * 60)
print("profile 정보가 없는 고객 이벤트 비율")
print("=" * 60)
print(f"고객 정보가 없는 이벤트 수: {missing_event_count}")
print(f"전체 이벤트 수: {total_event_count}")
print(f"비율: {missing_event_ratio:.2f}%")
print()

print("[예시 데이터]")
display(df_profile_missing_events.head())

profile 정보가 없는 고객 이벤트 요약
사용한 profile 컬럼: ['gender', 'age', 'income', 'became_member_on']
이벤트 행 수: 33749


고객 수: 2175

[event별 개수]
event
transaction        14996
offer received      9776
offer viewed        7865
offer completed     1112
Name: count, dtype: int64

profile 정보가 없는 고객 이벤트 비율
고객 정보가 없는 이벤트 수: 33749
전체 이벤트 수: 306137
비율: 11.02%

[예시 데이터]


,customer_id,event,time,offer_id,amount,event_reward,gender,age,income,became_member_on
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN,NaN,NaN,NaT
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN,NaN,NaN,NaT
6,c4863c7985cf408faee930f111475da3,offer received,0,2298d6c36e964ae4a3e7e9706d1fb8c2,NaN,NaN,NaN,NaN,NaN,NaT
10,744d603ef08c4f33af5a61c8c7628d1c,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT


df_transcript_profile 기준으로 확인한 결과,\
**profile 정보가 없는 고객 이벤트는 33,749건으로 전체 이벤트의 약 11.02%였다.**

이들은 단순 결측치라기보다 profile 전처리 과정에서 제외된 고객이\
left join 이후 미매칭 상태로 남은 경우로 해석할 수 있다.

또한 해당 이벤트에는 transaction, offer received, offer viewed, offer completed가 모두 포함되어 있어,\
이를 일괄 제거할 경우 퍼널 분석, 전환율 분석, 오퍼 반응 분석 결과가 왜곡될 가능성이 있다.

따라서 해당 이벤트를 삭제하지 않고 유지하되,\
고객 정보 보유 여부를 구분하는 has_profile 또는 customer_info_group='Unknown'과 같은 라벨링 컬럼을 추가해\
별도로 관리하는 방식이 적절하다.

결론\
transcript 이벤트는 유지\
profile 정보가 없는 고객은 has_profile 파생컬럼을 생성해 라벨링으로 관리

전체 행동 분석에서는 포함\
고객 특성 분석에서는 제외하거나 별도 집단으로 분리

In [39]:
profile_cols = ['gender', 'age', 'income', 'became_member_on']

# 1 = 고객 정보 있음, 0 = 고객 정보 없음
df_transcript_profile['has_profile'] = (
    df_transcript_profile[profile_cols].notna().all(axis=1).astype('uint8')
)

# 1 = 고객 정보 있음, 0 = 고객 정보 없음
df_transcript_portfolio_profile['has_profile'] = (
    df_transcript_portfolio_profile[profile_cols].notna().all(axis=1).astype('uint8')
)

display(df_transcript_profile.head(2))
display(df_transcript_portfolio_profile.head(2))

,customer_id,event,time,offer_id,amount,event_reward,gender,age,income,became_member_on,has_profile
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN,NaN,NaN,NaT,0


,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,has_profile
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaT,0


## 통합 csv 파일 보고서


### 오퍼 성과 분석 → transcript_portfolio
1. 행수 증가 없음
2. customer_id 중복 존재
    - transcript는 이벤트 로그 테이블로 한 고객이 여러 번 등장하는 게 정상
    - 예시:\
    offer received/offer viewed/transaction/offer completed 를 여러 번 하면 그만큼 행이 생긴다.

#### transcript_portfolio 컬럼 해설
- df_transcript_portfolio
    - 역할: 고객 이벤트 로그에 프로모션 정보(portfolio)를 결합한 테이블
    - 컬럼 해설
        - customer_id : 고객을 구분하는 고유 ID
        - event : 고객 행동 또는 오퍼 반응 이벤트 유형
        - time : 데이터 내 상대 시간 정보
        - offer_id : 관련 프로모션의 고유 ID
        - amount : 거래 발생 시 구매 금액
        - event_reward : 오퍼 완료 이벤트에서 지급된 실제 보상 금액
        - offer_type : 프로모션 유형
        - offer_reward : 프로모션 완료 시 제공되는 보상 금액
        - difficulty : 보상을 받기 위해 필요한 최소 지출 금액
        - duration : 프로모션 유효 기간
        - channels : 프로모션이 전달된 채널 목록
        - web : 웹 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - email : 이메일 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - mobile : 모바일 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - social : 소셜 채널 포함 여부 (0=정보 없음, 1=정보 있음)

### 고객 특성 분석 → transcript_profile
1. 행수 증가 없음
2. offer_id 중복 존재
    - 오퍼는 10종류 이고 그 오퍼가 여러 고객에게 여러 번 등장,\
    로그 테이블에서 offer_id가 계속 반복되는 게 정상

#### transcript_profile 컬럼 해설
- df_transcript_profile
    - 역할: 고객 이벤트 로그에 고객 정보(profile)를 결합한 테이블
    - 컬럼 해설
        - customer_id : 고객을 구분하는 고유 ID
        - event : 고객 행동 또는 오퍼 반응 이벤트 유형
        - time : 데이터 내 상대 시간 정보
        - offer_id : 관련 프로모션의 고유 ID
        - amount : 거래 발생 시 구매 금액
        - event_reward : 오퍼 완료 이벤트에서 지급된 실제 보상 금액
        - gender : 고객 성별 정보
        - age : 고객 나이 정보
        - income : 고객 소득 정보
        - became_member_on : 고객 가입일 정보. 신규/기존 고객 구분, 가입 기간 계산, 코호트 분석에 활용
        - has_profile : 고객 프로필 정보 존재 여부를 나타내는 변수 (0=정보 없음, 1=정보 있음)

### 세그먼트 / 종합 분석 → transcript_portfolio_profile
1. 행수 증가 없음
2. customer_id / offer_id 불일치가 이미 없었음
3. 결측 패턴도 자연스러움
    - offer_id, offer_type, offer_reward, difficulty, duration, channels\
    transaction은 원래 특정 offer와 직접 연결되지 않는 행이 많음
3. 중복 확인 결과\
한 고객은 여러 이벤트를 가진다\
같은 오퍼가 여러 고객, 여러 시점에 반복 등장함

#### transcript_portfolio_profile 컬럼 해설
- df_transcript_portfolio_profile
    - 역할: 고객 이벤트 로그에 프로모션 정보(portfolio)와 고객 정보(profile)를 함께 결합한 통합 테이블
    - 컬럼 해설
        - customer_id : 고객을 구분하는 고유 ID
        - event : 고객 행동 또는 오퍼 반응 이벤트 유형
        - time : 데이터 내 상대 시간 정보
        - offer_id : 관련 프로모션의 고유 ID
        - amount : 거래 발생 시 구매 금액
        - event_reward : 오퍼 완료 이벤트에서 지급된 실제 보상 금액
        - offer_type : 프로모션 유형
        - offer_reward : 프로모션 완료 시 제공되는 보상 금액
        - difficulty : 보상을 받기 위해 필요한 최소 지출 금액
        - duration : 프로모션 유효 기간
        - channels : 프로모션이 전달된 채널 목록
        - web : 웹 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - email : 이메일 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - mobile : 모바일 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - social : 소셜 채널 포함 여부 (0=정보 없음, 1=정보 있음)
        - gender : 고객 성별 정보
        - age : 고객 나이 정보
        - income : 고객 소득 정보
        - became_member_on : 고객 가입일 정보. 가입 기반 세그먼트, 코호트 분석, 가입 후 경과 기간 계산에 활용
        - has_profile : 고객 프로필 정보 존재 여부를 나타내는 변수 (0=정보 없음, 1=정보 있음)

> 채널 정보(web, email, mobile, social)를 별도로 넣지 않은 이유\
web, email, mobile, social은 channels 컬럼을 0/1 형태로 나눈 파생 컬럼이므로,\
원본 의미 설명에서는 중복을 피하기 위해 별도 설명을 생략했다.

# 저장

In [40]:
# ============================================================
# 통합 파일 저장
# ============================================================
# 이벤트 + 고객 정보
df_transcript_profile.to_csv("data/transcript_profile.csv", index=False)

# 이벤트 + 오퍼 정보
df_transcript_portfolio.to_csv("data/transcript_portfolio.csv", index=False)

# 이벤트 + 오퍼 정보 + 고객 정보
df_transcript_portfolio_profile.to_csv("data/transcript_portfolio_profile.csv", index=False)

print("통합 csv 3종 저장 완료")

통합 csv 3종 저장 완료


### 피드백
전체적으로 너무 완성도 높은 전처리 입니다.
위 코멘트 한 부분과 아래 내용 같이 검토해서 수정 후 EDA 다시 진행해 주세요.

* transaction 해석 기준 (가장 중요)
반드시 명시 필요
transaction은 특정 offer와 직접 매핑되지 않으므로,
“프로모션 성과”가 아닌 “기간 내 발생한 구매 행동”으로 해석해야 한다.

* instance 개념 접근(해석, 언급)
동일 고객이 동일 offer를 여러 번 받을 수 있으므로,
(customer_id + offer_id)는 하나의 고유 단위가 아니며,
필요 시 instance 단위로 분석해야 한다.

* duration(유효기간) 중요성
분석 왜곡 방지 핵심
프로모션 반응(viewed, completed, transaction 등)은
반드시 duration(유효기간) 내에서 해석해야 한다.

* 이벤트 구조 한 줄 정리 (전체 연결용)
데이터는
received → viewed → transaction(반복 가능) → completed
흐름으로 해석하는 것이 명시적으로 인식되고 설명 되어야 한다.